# Physical AI 基础篇 智能机器人课程 2026 Spring 第6讲练习资料

本讲围绕真实机器人数据的质量检查、时间同步、格式转换和标签审计展开。你会把 ROS bag 中的多路消息整理成可供 LeRobotDataset 使用的训练数据，并观察同步偏移如何影响后续策略。

完整版包含完整参考实现；填空版从自己的源文件生成，只保留原有的 5 个 TODO 差异。


### 开始：格式与 Colab 使用方式

按 cell 顺序执行。安装和数据准备只需在 Colab 中进行；重型模型与 checkpoint 会在实际使用它们的 cell 中按需下载。遇到长时间下载时请保持 session 连接，并留意每个步骤的 self-check。


In [ ]:
# cell 0-1: 安装库（Colab 仅在 Colab 中执行。5约需几分钟）
import sys

if "google.colab" in sys.modules:
    # 固定版个个个的原因:
    #   rosbags>=0.10.9 : MCAP 格式的 bag 读取 0.10.9 在该版个个个以后稳定
    #   lerobot==0.5.1  : 固定为与第 1 讲相同的版个个个(API版个个間，大宽度変为了)
    #   robosuite/mujoco: Step 5 的实时 rollout 用(robosuite 1.5系列列 + mujoco 3.9以下)
    #   transformers==5.3.0 : lerobot 0.5.1  extras 指定的版个个个。5.4.0 ， PretrainedConfig 
    #        dataclass 已变为、lerobot 0.5.1 的 GR00TN15Config  import 时 TypeError 会变成。
    #        lerobot 是 transformers  base 因为不是直接依赖、如果不固定， Colab 环境中的新版个个个会保留。
    #        ACT 是 transformers 不使用它，因此正文・Step5・Step6 不会受影响。
    # 注意意意: robomimic 绝对无需 unzip安装。robomimic 是 diffusers==0.11.1 固定后、
    #        lerobot 的 diffusers>=0.27.2 冲突并、破坏环境。
    !pip install -q "rosbags>=0.10.9" "lerobot==0.5.1" "transformers==5.3.0"                     "robosuite>=1.5,<2" "mujoco<3.10"

    # numpy 被替换时需要重启 runtime(重启后从此 cell 重做)
    import importlib
    import numpy
    importlib.reload(numpy)
print("install: OK")

In [ ]:
# cell 0-2: 获取辅助模块和练习几据(正文约 65 MB・1〜2 分)
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from huggingface_hub import hf_hub_download
    # 每次 HF 从 HF 获取最新版个个个(force_download)。utils 更新后只需重新上传即可
    # 同步、notebook 与 utils.py 的避免版个个个不一致。
    for _f in ("utils.py", "utils_rollout.py"):
        hf_hub_download("Adwaver4157/pai2026-lecture6-data", _f, repo_type="dataset", local_dir=".", force_download=True)

import utils

# lerobot × transformers 兼容性保护（Colab 较新的 transformers 使其无害化并、
# lerobot.policies 先通过一次）。如果这里失败， setup 可把问题限制在 setup 中。
utils.ensure_lerobot_importable()

# 正文（Step 0–4）使用的约 65 MB 只获取。
# 大型训练模型(Step 5 的实时 rollout / 拓展 Step 6)是、
# 在使用它的 cell 需要时自动追加下载。
DATA = utils.fetch_data()
print("几据位置:", DATA.resolve())

In [ ]:
# cell 0-3: setup 的 self-check
utils.check_step0()

## Step 1：Inspect——解剖原始数据（5 分钟）

先从 topic 频率、时间戳和设备延迟入手。ROS bag 同时记录设备写入的 `header.stamp` 与记录机收到消息的时间；这两个时间轴不一致时，单纯看消息数量可能会错过真正的问题。


In [ ]:
# cell 1-1(与代码): 全部部 bag 的 topic 表
# 4 topic: 相机2(20Hz) / 关节状态(50Hz) / 遥操作指令(20Hz)
rate_df = utils.topic_table()
rate_df.pivot_table(index="bag", columns="topic", values="rate_hz")

In [ ]:
# cell 1-2【TODO-1】: 1 个 bag 的时间戳绘制散点图
# 提示: topics[topic 名称称称]["stamp"]  header.stamp(秒)的几组
import matplotlib.pyplot as plt

bag = utils.list_bags()[0]          # ep_000
topics = utils.read_bag(bag)

fig, axes = plt.subplots(1, 2, figsize=(12, 3), sharey=True)
for ax, xlim in zip(axes, [None, 0.5]):
    for row, topic in enumerate(utils.ALL_TOPICS):
        # ---- TODO 这里从(2 行): stamp 取出几组并、转换为相对于 episode 开头的时间
        stamps = ...
        t_rel = ...
        # ---- TODO 这里到
        ax.scatter(t_rel, [row] * len(t_rel), s=4, label=topic.split("/")[1])
    ax.set_yticks(range(4), [t.split("/")[1] for t in utils.ALL_TOPICS])
    ax.set_xlabel("time since start [s]")
    if xlim:
        ax.set_xlim(0, xlim)
        ax.set_title("first 0.5 s (zoom)")
axes[0].set_title(f"{bag.name}: message timestamps")
plt.tight_layout(); plt.show()

In [ ]:
#@title ▶ 答案 1(詰開执行并请) { display-mode: "form" }
import matplotlib.pyplot as plt

bag = utils.list_bags()[0]
topics = utils.read_bag(bag)
fig, axes = plt.subplots(1, 2, figsize=(12, 3), sharey=True)
for ax, xlim in zip(axes, [None, 0.5]):
    for row, topic in enumerate(utils.ALL_TOPICS):
        stamps = topics[topic]["stamp"]
        t_rel = stamps - topics[utils.TOPIC_CAM_AGENT]["stamp"][0]
        ax.scatter(t_rel, [row] * len(t_rel), s=4)
    ax.set_yticks(range(4), [t.split("/")[1] for t in utils.ALL_TOPICS])
    ax.set_xlabel("time since start [s]")
    if xlim:
        ax.set_xlim(0, xlim); ax.set_title("first 0.5 s (zoom)")
axes[0].set_title(f"{bag.name}: message timestamps")
plt.tight_layout(); plt.show()

In [ ]:
# cell 1-3(与代码): bag按・按 topic 的 median(stamp − 接收时间) [ms]
# 「stamp 是设备写入的时间、接收时间是记录机PC收到的时间」——正常情况下
# stamp 应稍早(= −几 ms)应该如此。大宽度为正(未来)偏移 bag 是?
latency_df = utils.latency_table()
ax = latency_df.plot.bar(figsize=(12, 3), ylabel="median(stamp - recv) [ms]")
ax.axhline(0, color="k", lw=0.5)
latency_df.round(1)

### 写下你的观察（1 行即可）

记录哪一个 topic 或 bag 最值得进一步检查，以及你依据的时间戳/频率证据。


In [ ]:
# cell 1-5: Step 1 的 self-check
utils.check_step1(rate_df, latency_df)

## Step 2：Sync——自己实现 ApproximateTime（8 分钟）

以主相机帧为基准，在允许的 `slop` 内为其他 topic 找最近消息。关键是区分“没有匹配”与“勉强匹配”，并检查匹配残差；同步后的数据质量取决于真实时间关系，而不是只取决于消息计数。


In [ ]:
# cell 2-1【TODO-2】: 最近邻匹配 nearest() 完成(这是核心)
import numpy as np

def nearest(stamps, t, slop):
    """升序的 stamps 从 t 返次距离最近时间的索引。
    |stamps[i] - t| > slop 若 None(无匹配)。"""
    # ---- TODO 这里从(3 行): searchsorted 用 searchsorted 求插入点并、前后 2 选择较近的候选
    i = ...
    cands = ...
    best = ...
    # ---- TODO 这里到
    return best if abs(stamps[best] - t) <= slop else None
    # 另一种解法: np.argmin(np.abs(stamps - t)) 的 1 行也正确(几据量大时较慢)


def sync_bag(topics, slop=0.020):
    """agentview 以帧时间为基准同步所有 topic。"""
    cam = topics[utils.TOPIC_CAM_AGENT]
    jnt = topics[utils.TOPIC_JOINTS]
    rows = {"t": [], "state": [], "action": [], "cam_idx": [], "wrist_idx": []}
    for i, t in enumerate(cam["stamp"]):
        iw = nearest(topics[utils.TOPIC_CAM_WRIST]["stamp"], t, slop)
        ia = ...   # TODO(1 行): 指令 /leader/action 也 wrist 与同様到匹配
        if iw is None or ia is None:
            continue
        # 关节使用插值——但超出范围则丢弃(与代码。D1 已处理端点 clamp 的陷阱)
        if not (jnt["stamp"][0] <= t <= jnt["stamp"][-1]):
            continue
        state = np.array([np.interp(t, jnt["stamp"], jnt["val"][:, j]) for j in range(7)])
        rows["t"].append(t); rows["state"].append(state)
        rows["action"].append(topics[utils.TOPIC_ACTION]["val"][ia])
        rows["cam_idx"].append(i); rows["wrist_idx"].append(iw)
    return {k: np.asarray(v) for k, v in rows.items()}


topics = utils.read_bag(utils.list_bags()[0])
synced = sync_bag(topics)
print(f"采用帧几: {len(synced['t'])}(相机 {len(topics[utils.TOPIC_CAM_AGENT]['stamp'])} 帧中)")

In [ ]:
#@title ▶ 答案 2(詰開执行并请) { display-mode: "form" }
import numpy as np

def nearest(stamps, t, slop):
    i = np.searchsorted(stamps, t)
    cands = [j for j in (i - 1, i) if 0 <= j < len(stamps)]
    best = min(cands, key=lambda j: abs(stamps[j] - t))
    return best if abs(stamps[best] - t) <= slop else None

def sync_bag(topics, slop=0.020):
    cam = topics[utils.TOPIC_CAM_AGENT]
    jnt = topics[utils.TOPIC_JOINTS]
    rows = {"t": [], "state": [], "action": [], "cam_idx": [], "wrist_idx": []}
    for i, t in enumerate(cam["stamp"]):
        iw = nearest(topics[utils.TOPIC_CAM_WRIST]["stamp"], t, slop)
        ia = nearest(topics[utils.TOPIC_ACTION]["stamp"], t, slop)
        if iw is None or ia is None:
            continue
        if not (jnt["stamp"][0] <= t <= jnt["stamp"][-1]):
            continue
        state = np.array([np.interp(t, jnt["stamp"], jnt["val"][:, j]) for j in range(7)])
        rows["t"].append(t); rows["state"].append(state)
        rows["action"].append(topics[utils.TOPIC_ACTION]["val"][ia])
        rows["cam_idx"].append(i); rows["wrist_idx"].append(iw)
    return {k: np.asarray(v) for k, v in rows.items()}

topics = utils.read_bag(utils.list_bags()[0])
synced = sync_bag(topics)
print(f"采用帧几: {len(synced['t'])}")

In [ ]:
# cell 2-2: Step 2 的 self-check(实现差异 ±3 帧以内都允许)
utils.check_step2("ep_000", len(synced["t"]))

In [ ]:
# cell 2-3【TODO-3】: slop  {5, 20, 100} ms 扫描并观察采用率
# 同时绘制「匹配残差」(已采用消息的 |Δt|)直方图
import matplotlib.pyplot as plt

slops = [0.005, 0.020, 0.100]
bags = utils.list_bags()
rates = {s: [] for s in slops}
for bag in bags:
    tp = utils.read_bag(bag)
    n_cam = len(tp[utils.TOPIC_CAM_AGENT]["stamp"])
    for s in slops:
        # ---- TODO 这里从(1 行): slop=s 同步时的采用率(采用帧几 ÷ n_cam)
        rate = ...
        # ---- TODO 这里到
        rates[s].append(rate)

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
x = np.arange(len(bags))
for k, s in enumerate(slops):
    axes[0].bar(x + 0.27 * k, rates[s], width=0.25, label=f"slop {int(s*1e3)} ms")
axes[0].set_xticks(x + 0.27, [b.name for b in bags], rotation=90)
axes[0].set_ylabel("adoption rate"); axes[0].legend(fontsize=8)

# 残差直方图(ep_000): slop 可看到超出阈值的候选分布
tp = utils.read_bag(bags[0])
cam, wr = tp[utils.TOPIC_CAM_AGENT]["stamp"], tp[utils.TOPIC_CAM_WRIST]["stamp"]
res = [abs(wr[nearest(wr, t, 1.0)] - t) * 1e3 for t in cam]
axes[1].hist(res, bins=40)
for s in slops:
    axes[1].axvline(s * 1e3, color="r", ls="--", lw=0.8)
axes[1].set_xlabel("nearest-match residual [ms] (ep_000, wrist)"); axes[1].set_ylabel("count")
plt.tight_layout(); plt.show()

print("观察: slop 5→20→100 ms 时采用率如何变化?")
print("      出现延迟异常的 bag 的采用率…下降了吗?(→ 不会下降!)")

In [ ]:
# cell 2-4(与代码): 用物理证据交叉验证——指令与关节响应的互相关
# 「stamp 不盲信」检测方法。指令(速度指令宽度度)与关节速度波形
# 逐步错开并叠加、重合度最高的偏移量 = 有效时间偏移、视为。
# 实现已通过 utils.xcorr_lag_ms(插值网格化 + 平滑 + 一阶差分增强鲁棒性)。
lag_by_bag = {}
for bag in utils.list_bags():
    tp = utils.read_bag(bag)
    lag_by_bag[bag.name] = utils.xcorr_lag_ms(tp)
    print(f"{bag.name}: lag = {lag_by_bag[bag.name]:+7.1f} ms")

utils.check_sync(lag_by_bag)

### 填写思考（2 个问题）

1. 增大 `slop` 后，保留率和匹配残差分别如何变化？
2. 设备写入时间与物理响应之间不一致时，你会相信哪个证据，为什么？


In [ ]:
#@title ⏩ Step 2 checkpoint(用于跟上进度: 加载参考同步结果) { display-mode: "form" }
# 自己实现的 synced 已有的同学无需执行。执行也只会覆盖，不会造成影响。
synced = utils.load_synced("ep_000")
print("checkpoint 加载完成:", {k: v.shape for k, v in synced.items()})

## Step 3：Format & Visualize——转换为 LeRobotDataset 并核验（5 分钟）

把同步后的多路数据写成 LeRobotDataset。除了 shape 和 dtype，还要检查 episode 长度、动作维度、视频帧和夹爪动作是否彼此对齐。


In [ ]:
# cell 3-1【TODO-4】: features 填写字典值并 3 个转换
import shutil
from lerobot.datasets.lerobot_dataset import LeRobotDataset

root3 = "my_dataset_3ep"
shutil.rmtree(root3, ignore_errors=True)   # 重复执行保护

features = {
    # ---- TODO 这里从: 值(dtype / shape / names)填写。无需 unzip改变 key
    # 提示: 图像是 dtype "video"、shape 是 (高度, 宽度, 通道)。
    #         state 与 action 是两者也 float32 的 7 维
    "observation.images.agentview": {"dtype": ..., "shape": ..., "names": ["height", "width", "channels"]},
    "observation.images.wrist":     {"dtype": ..., "shape": ..., "names": ["height", "width", "channels"]},
    "observation.state": {"dtype": ..., "shape": ..., "names": [f"joint{i}" for i in range(1, 8)]},
    "action":            {"dtype": ..., "shape": ...,
                          "names": ["dx", "dy", "dz", "drx", "dry", "drz", "grip"]},
    # ---- TODO 这里到
}

ds3 = LeRobotDataset.create(repo_id="student/lecture6-3ep", fps=20,
                            features=features, root=root3)

import json
instr = json.loads((utils.DATA_DIR / "bags" / "collection_log.json").read_text())
for name in ["ep_000", "ep_004", "ep_010"]:
    d = utils.load_synced(name)
    for k in range(len(d["t"])):
        ds3.add_frame({
            "observation.images.agentview": d["agentview"][k],
            "observation.images.wrist": d["wrist"][k],
            "observation.state": d["state"][k],
            "action": d["action"][k].astype("float32"),
            "task": instr[name]["instruction"],
        })
    ds3.save_episode()
    print(f"{name}: {len(d['t'])} 帧转换")

# 最后务必 finalize() 调用(忘记调用会导致元几据不写入、
# 之后无法读取「损坏的几据集」——官方文档的 docs 的 Common Issues 首要问题)
# ds3.finalize()   # ← 请取消此注意意释
print("转换完成:", ds3.num_episodes, "episodes /", ds3.num_frames, "frames")

# 用于写入的对象不能直接用于读取，因此、重新打开
ds3 = LeRobotDataset(repo_id="student/lecture6-3ep", root=root3)

In [ ]:
# cell 3-1b: Step 3 的 self-check
utils.check_step3(ds3)

In [ ]:
# cell 3-2(与代码): 用视频确认转换后的 episode
# 确认点只有 1 一个: 关闭夹爪的指令(action 的 7 维)的上升沿与、
# 视频中的「抓取瞬间」一致(若不一致，请怀疑同步)
utils.episode_video(ds3, 0)

In [ ]:
# cell 3-3(与代码): 全部部 16 个几据的统计 dashboard
# 从这里开始 checkpoint 已转换的 16 个几据集(延迟下载)
from lerobot.datasets.lerobot_dataset import LeRobotDataset

ds16 = LeRobotDataset(repo_id="pai2026/lecture6-bags16",
                      root=utils.DATA_DIR / "lerobot" / "lecture6-bags16")
utils.stats_report(ds16)
print("问题: episode 长度和 jerk(运动抖动)显示出的 bag 是哪一个?")

### 写下你的观察

比较视频、动作和统计图，记录一个你认为需要保留、过滤或重新标注的现象。


## Step 4a：Curate——填写判断表（3 分钟 + 随时补充）

数据清洗不是“看到异常就删除”。对每个 episode 记录 `keep`、`filter` 或 `relabel`，并在 `method` 和 `reason` 中写下可复核的依据，例如延迟、最大时间差、jerk、目视结果或 kNN。这样下游训练才能追溯每个决定。


In [ ]:
# cell 4-1(与代码): 判断表模板
import pandas as pd

bag_names = [b.name for b in utils.list_bags()]
df = pd.DataFrame({
    "episode_id": range(len(bag_names)),
    "bag": bag_names,
    "decision": "keep",          # keep / filter / relabel 改写为
    "method": "",                # 检测方法(延迟 / max Δt / jerk / 目视 / kNN 等)
    "reason": "",                # 依据(带单位的几值更有说服力)
    "new_task": "",              # relabel 时才填写: lift 还是 can
})
# 示示示示例: Step 1 中发现的延迟异常 bag(请根据自己的结果改写)
# df.loc[6, ["decision", "method", "reason"]] = ["filter", "latency", "joint stamp +348ms"]
df

### 练习：填写判断表

- 把 Step 1（延迟）、Step 2（互相关）和 Step 3（统计）发现的问题写进对应行；
- 对犹豫的 bag 用 `utils.episode_video(ds16, episode_id)` 做**最终目视检查**；
- 课堂内填好**至少 1 行**即可，其余可在有时间时完成。


In [ ]:
# cell 4-3(与代码): 应用判断表并「交付物 1」创建
# relabel 是 lerobot 的 dataset_tools.modify_tasks 的薄 wrapper(in-place 因此
# 应用于原几据集的副个)
curated, keep_eps = utils.apply_curation(
    utils.DATA_DIR / "lerobot" / "lecture6-bags16", df, "my_dataset_curated")

## Step 4b：用 embedding kNN 审计标签——统计正常也可能有问题（教师演示）

*对应课程幻灯片“指令标签错误”和“经典工具能做到什么程度”。*

即使通过延迟、gap、长度和 jerk 的全部检查，仍可能有 **2 个 bag 不对**。数值统计和图表都正常，问题在于**指令标签与实际动作不匹配**，也就是 instruction label noise。

这里实现两种工具：

- **Deep kNN** [Bahri+ ICML 2020]：在 embedding 空间中让邻居投票，怀疑那些预测标签与原标签不一致的样本；
- **最近质心（prototype）**：用样本到各任务中心向量的距离查找不一致。

本练习的 lift 与 can 在视觉上容易区分，因此只用视觉 embedding 就能发现问题。但真实数据中，放入架子与从架子取出等任务可能很相似，需要考虑多视角和多模态证据；“应该相信哪个模态、模态失效时怎么办”仍是机器人领域几乎未解决的研究问题。请先亲手运行工具。


In [ ]:
# cell 4-5(与代码): 加载预计算 embedding 并进行「保留顺序的 pooling」
# vis_frames: 每个 episode 16 帧的 DINOv2 特征(384 维度)
emb = dict(np.load(utils.DATA_DIR / "checkpoints" / "embeddings.npz", allow_pickle=True))
ep_ids, labels = emb["ep_ids"].astype(str), emb["labels"].astype(str)
print("episode:", len(ep_ids), "/ 标签分布:", dict(zip(*np.unique(labels, return_counts=True))))

# 根据判断表 filter 的样个排除(curation 保护下游流程、的实示示示示例)
filtered_bags = set(df[df.decision == "filter"].bag)
keep = np.array([e not in filtered_bags for e in ep_ids])
print("kNN 对象:", int(keep.sum()), "个(filter 排除已处理的)")

# 平均 pooling vs segment pooling(前半‖后半)对比
# 平均 pooling 会「时间顺序」丢弃，因此、去程和次程会压到同一点
mean_pool = emb["vis_frames"].mean(1)
seg_pool = utils.segment_pool(emb["vis_frames"])

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, feats, title in [(axes[0], mean_pool, "mean pooling"),
                         (axes[1], seg_pool, "segment pooling (1st half || 2nd half)")]:
    x = utils.zscore(feats)
    # 2 投影到二维(PCA 相当的 SVD ，)
    u, s, vt = np.linalg.svd(x - x.mean(0), full_matrices=False)
    p = u[:, :2] * s[:2]
    for lab, m in [("lift", "o"), ("can", "^")]:
        sel = labels == lab
        ax.scatter(p[sel, 0], p[sel, 1], marker=m, label=f"label={lab}", s=60)
    for i, e in enumerate(ep_ids):
        ax.annotate(e[-3:], (p[i, 0], p[i, 1]), fontsize=7)
    ax.set_title(title); ax.legend()
plt.tight_layout(); plt.show()
print("在右图中、lift 虽然标为该标签却 can 是否混入另一侧?")

In [ ]:
# cell 4-6【TODO-5】: leave-one-out kNN 用多几投票列出可疑样个
feats = utils.zscore(utils.segment_pool(emb["vis_frames"]))[keep]
ids_k, labels_k = ep_ids[keep], labels[keep]

def knn_vote(D, labels, k=5):
    """距离矩阵 D 从 leave-one-out kNN 返次多几投票的预测标签。"""
    # ---- TODO 这里从(5 行):
    #   1. D 的对角线设为 inf (不把个身作为邻居 = leave-one-out)
    #   2. 每个 i 按距离从近到远 k 取出个标签(argsort)、
    #   3. 多几決(np.unique(..., return_counts=True))决定预测标签
    D = D.copy()
    ...
    pred = []
    for i in range(len(labels)):
        ...
        ...
        pred.append(...)
    # ---- TODO 这里到
    return np.asarray(pred)

D = np.linalg.norm(feats[:, None] - feats[None, :], axis=2)
pred = knn_vote(D, labels_k, k=5)
suspects = [str(e) for e in ids_k[pred != labels_k]]
print("kNN 预测标签与原标签不一致(=可疑样个):", suspects)
utils.check_labels(suspects)

In [ ]:
#@title ▶ 答案 5(詰開执行并请) { display-mode: "form" }
feats = utils.zscore(utils.segment_pool(emb["vis_frames"]))[keep]
ids_k, labels_k = ep_ids[keep], labels[keep]

def knn_vote(D, labels, k=5):
    D = D.copy()
    np.fill_diagonal(D, np.inf)
    pred = []
    for i in range(len(labels)):
        nn = labels[np.argsort(D[i])[:k]]
        vals, counts = np.unique(nn, return_counts=True)
        pred.append(vals[np.argmax(counts)])
    return np.asarray(pred)

D = np.linalg.norm(feats[:, None] - feats[None, :], axis=2)
pred = knn_vote(D, labels_k, k=5)
suspects = [str(e) for e in ids_k[pred != labels_k]]
print("可疑样个:", suspects)
utils.check_labels(suspects)

In [ ]:
#@title (拓展・仅供阅读OK) 2 多模态投票与「要目视」流程 { display-mode: "form" }
# 不仅视觉，还对关节轨迹(64 step × 7 关节)也运行相同的 Deep kNN 。
# 两种模态都判为可疑 bag 可确定为可疑样个、票割 bag 是「要目视」交给——
# 这是处理流程。「该相信哪个模态」是课程中提到的未解决问题、
# 这里体验最简单的解法(多几決 + 升级给人工)。
# 想继续深入的同学可: Confident Learning [Northcutt+ JAIR 2021](pip install cleanlab)
# 相同 embedding + 应用到轻量分类器并、kNN 与 kNN 检测结果比较。
feats_j = utils.zscore(emb["joint"])[keep]
Dj = np.linalg.norm(feats_j[:, None] - feats_j[None, :], axis=2)
suspects_j = {str(s) for s in ids_k[knn_vote(Dj, labels_k, k=5) != labels_k]}
both = sorted(set(suspects) & suspects_j)
split = sorted(set(suspects) ^ suspects_j)
print("关节模态可疑样个:", sorted(suspects_j))
print("两种模态一致的可疑样个:", both)
print("票割(=交给人工目视)bag:", split if split else "无")

In [ ]:
# cell 4-8: 目视检查可疑样个 → relabel
# (1) 并排查看可疑样个的视频和指令
import json
instr = json.loads((utils.DATA_DIR / "bags" / "collection_log.json").read_text())
name = suspects[0]                       # 2 第一个使用 suspects[1] 改为后重新执行
ep_id = [b.name for b in utils.list_bags()].index(name)
print(f"{name} 的指令:「{instr[name]['instruction']}」← 是否与视频一致?")
utils.episode_video(ds16, ep_id)

In [ ]:
# (2) 判断表 relabel 更新后、完成 curated 几据集
for name in suspects:
    ep_id = [b.name for b in utils.list_bags()].index(name)
    df.loc[ep_id, ["decision", "method", "new_task"]] = ["relabel", "kNN+目视", "can"]
    df.loc[ep_id, "reason"] = "视频是 can 的 pick&place、指令是 lift"

curated, keep_eps = utils.apply_curation(
    utils.DATA_DIR / "lerobot" / "lecture6-bags16", df, "my_dataset_curated")
df   # 查看完成的判断表并、确认自己能否说明依据

## Step 5：训练对比——核对交付物（4 分钟，以阅读为主）

*对应课程幻灯片“数据质量与策略性能”。*

这里准备了 3 个 checkpoint：它们使用相同的 ACT 和相同的训练规模（200k step），只改变数据（can 任务，完整数据预训练）：

1. **① 保留同步偏移**：与 D1 一样带有 +350 ms 偏移（300 条）；
2. **② 修正同步**：只修复时间同步，但保留“动作笨拙但成功”的 worse 样本（300 条）；
3. **③ 同时过滤 worse**：修正同步并删除 worse（仅 200 条）。

重点看两件事：① 的 loss 仍然会正常下降，说明“能训练起来”和“机器人能正确运动”是两回事；② 与 ③ 的排名则检验“应该丢弃低质量数据”的直觉。在这个规模下，过滤可能损失数据量和覆盖范围，因此处理是否正确依赖任务、规模和策略，必须测量副作用。


In [ ]:
# cell 5-1(与代码): 每个条条成功率和 rollout 视频(预计算)
import json
from pathlib import Path
from IPython.display import Video, display

res_path = utils.DATA_DIR / "checkpoints" / "step5_results.json"
if res_path.exists():
    res = json.loads(res_path.read_text())
    import pandas as pd
    tbl = pd.DataFrame(res["conditions"])
    display(tbl[["name", "description", "n_episodes", "n_rollouts", "success_rate"]])
    # 训练曲线(①也 loss 会下降、进行观察)
    from IPython.display import Image as IPImage
    curve = utils.DATA_DIR / "checkpoints" / "step5_loss_curves.png"
    if curve.exists():
        display(IPImage(str(curve)))
    for mp4 in sorted((utils.DATA_DIR / "checkpoints").glob("step5_*.mp4"))[:2]:
        print(mp4.name)
        display(Video(str(mp4), embed=True, width=400))
else:
    print("(预计算结果会在前一天加入发布几据)")

In [ ]:
#@title (可选・如果有时间) 最佳条条②的 checkpoint 做 live rollout 1 个 { display-mode: "form" }
# CPU ， 1〜2 分钟。繁忙时请跳过(运行的内容与上面视频相同)。
try:
    utils.fetch_data(["checkpoints/act_best/*"])   # 約 198 MB(首下一次)
    ckpt = utils.DATA_DIR / "checkpoints" / "act_best"
    if not ckpt.exists():
        print("未发布 checkpoint，跳过")
    else:
        import mujoco  # noqa: F401
        print("rollout 开始…(几分)")
        # 实现位于发布的 utils_rollout.py(robosuite 直接启动)
        import utils_rollout
        utils_rollout.rollout_once(ckpt)
except Exception as e:
    import traceback
    print(f"此环境跳过 live rollout 跳过执行({e})")
    traceback.print_exc()   # GL 便于当天的助教追踪配置错误等原因 TA 

### 讨论（与教师一起）

- **①（不修同步）与 ②（修同步）的差异**：350 ms 偏移破坏了多少策略性能？结合发展实验的 0/100/200 ms 剂量反应，说明可接受的偏移范围。
- **②（含 worse 的 300 条）与 ③（过滤后 200 条）的差异**：低质量数据并不总该删除。lift 的小实验里 worse 可能无害，而 can 在这个规模下删除它会损失覆盖范围；“先测副作用，再决定过滤或重标注”是今天的原则。
- 两个标签错误样本没有影响今天 ACT 的成功率，因为 ACT 没有把语言指令作为输入。什么样的策略会从 relabel 中受益？


## Step 6（拓展小实验）：同步偏移从多少毫秒开始破坏策略

这是一个把“数据质量 → 成功率”亲手连起来的实验（约 15–20 分钟）。Step 5 使用的是教师预训练结果；这里自己训练一个最小版本：

- 任务是 lift（约 2.5 秒），固定 20 条演示，图像 96×96，小型 ACT，5000 step；
- 唯一质量旋钮是“把所有 episode 的观测与动作错开多少毫秒”：`{0, 100, 200}`；
- 每个数据集约 1 MB，Colab T4 GPU 训练约 10–12 分钟。请选择“运行时 → 更改运行时类型 → T4 GPU”。

> ⚠️ 更改运行时类型会初始化环境，pip 安装和补丁都会消失。切换到 T4 后，请先重新执行 cell 0-1、0-2，再执行 6-1。普通“重启 session”会保留文件，不必重新安装。
>
> **恢复步骤**：①运行时 → 重启 session；②重新执行 0-1、0-2；③确认 0-2 中 `import lerobot.policies` 成功；④回到 6-1。

三种条件任选一种训练和评估，再与教师曲线比较：

| 偏移 | 0 ms | 100 ms | 200 ms |
|---|---:|---:|---:|
| 教师实测（5000 step、每项 100 次 rollout） | 80% | 74% | 56% |

200 ms 的下降具有统计显著性（p<0.001），100 ms 只有趋势、差异不显著。你自己的 10 次 rollout 噪声较大；全班汇总后才会看出稳定趋势。


In [ ]:
# cell 6-1: 选择条条并训练(T4 ，5000steps约3小时。checkpoint 存在时跳过)
SHIFT_MS = 0  #@param [0, 100, 200] {type:"raw"}

import torch
assert torch.cuda.is_available(),     "GPU runtime到请切换(runtime→更改 runtime 类型→T4 GPU)。"     "切换后 0-1, 0-2 重新执行后重新次到这里。"

from pathlib import Path
name = f"lift-sync{SHIFT_MS:03d}ms"
utils.fetch_data([f"lerobot/{name}/*"])              # 训练几据 約 1 MB
ckpt = Path(f"mini_train_{name}/checkpoints/last/pretrained_model")

if ckpt.exists():
    print(f"训练好的 checkpoint 存在: {ckpt}")
elif not (utils.DATA_DIR / "lerobot" / name).exists():
    print(f"几据集 {name} 未找到。cell 0-2 请重新执行。")
else:
    # 与学生运行的相同小型 ACT 配置(教师实测: T4 相当环境下 5000 step ≈ 10 分)
    args = " ".join([
        f"--dataset.repo_id=pai2026/{name}",
        f"--dataset.root={utils.DATA_DIR}/lerobot/{name}",
        "--policy.type=act", "--policy.chunk_size=20", "--policy.n_action_steps=20",
        "--policy.dim_model=256", "--policy.n_encoder_layers=2", "--policy.n_decoder_layers=1",
        "--policy.dim_feedforward=1024", "--policy.device=cuda", "--policy.push_to_hub=false",
        "--steps=200", "--batch_size=16", "--num_workers=2", "--log_freq=500", "--save_freq=5000",
        f"--output_dir=mini_train_{name}", "--wandb.enable=false",
    ])
    import subprocess
    ret = subprocess.run(f"lerobot-train {args}", shell=True)
    # !lerobot-train 是 shell 执行，即使失败 cell 也可能显示正常结束。显式检测。
    if ret.returncode != 0:
        raise RuntimeError(
            f"lerobot-train 失败(exit {ret.returncode})。请检查上面的日志。"
            "通常可以从 0-2 的 import 发现。若仍失败则重启 session→0-1 从。"
        )

In [ ]:
# cell 6-2: 评估(10 rollouts、几分)— 自己训练的策略 or 教师的 checkpoint
from pathlib import Path
import utils_rollout

name = f"lift-sync{SHIFT_MS:03d}ms"
ckpt = Path(f"mini_train_{name}/checkpoints/last/pretrained_model")
if not ckpt.exists():
    utils.fetch_data([f"checkpoints/mini/{name}/*"])   # 教师的训练好的 約 65 MB(首下一次)
    ckpt = utils.DATA_DIR / "checkpoints" / "mini" / name
    print(f"(自己的 checkpoint 不存在，因此使用教师训练的版个个: {ckpt})")

rate = utils_rollout.rollout(ckpt, task="lift", n=10)
print()
print(f"你的条条: 偏移 {SHIFT_MS} ms → 成功率 {rate:.0%}")
print("教师实测曲线(80% / 74% / 56%)与之相比如何?")

### 记录区

> 我负责的偏移：____ ms；成功率：____%（10 次 rollout）；与教师曲线的差异：____
> 观察（例如：100 ms 与 Step 1 测得的相机传输延迟处在同一数量级）：

**思考问题**：本实验中 200 ms 偏移让成功率下降约 24 个百分点（100 ms 约 6 个百分点，仍在误差范围内）。结合 Step 1 的延迟和 Step 2 的 `slop`，你的机器人记录系统大约能容忍多大偏移？


## 想继续探索（拓展，不要求提交）

- 先完成上面的 Step 6，至少尝试一个条件；也可以与同学分工，合并出三点曲线；
- 用自己的规则对完整数据做清洗并重新训练：对 can 的完整数据（ph 200 条 + multi-human 300 条，已转换并提供）按长度、jerk、延迟阈值执行规则，再训练 ACT 并评估，看看能否超过教师条件②的 88%；
- 用偏移估计“拯救”D1 bag，重新同步后加入训练集，成功率会提高吗？
- 测试加入或移除 multi-human 数据中 okay 层的敏感性，观察数据量与质量之间的权衡。
